# ACADIA 2023 - GML Day 02 Topologic Tutorial

## Advanced Topology Operations with topologic_fast

This notebook is an adaptation of the ACADIA 2023 GML Day 02 topologicpy tutorial for the `topologic_fast` library.

### Topics Covered:
- Creating a CellComplex building structure
- Assigning categories to cells based on position
- Creating and visualizing graphs
- Computing depth maps
- Visualizing buildings with color-coded faces

In [ ]:
# Import libraries
import topologic_fast as tf
import plotly.graph_objects as go
import plotly.express as px
import math
import numpy as np

print("Done importing libraries")

## 1. Create a Building CellComplex

We'll create a 10-story building divided into cells (rooms/floors).

In [ ]:
# Create a building structure
# Parameters
building_width = 10
building_length = 10
floor_height = 3
num_floors = 10
divisions_x = 2  # rooms per floor in X
divisions_y = 2  # rooms per floor in Y

# Create cells for each room
cells = []
cell_categories = []  # Store category for each cell

room_width = building_width / divisions_x
room_length = building_length / divisions_y

for floor in range(num_floors):
    z = floor * floor_height
    
    # Determine category based on height
    if z < 6:
        cat = 0  # Ground level
    elif z < 12:
        cat = 1  # Low-rise
    elif z < 18:
        cat = 2  # Mid-rise lower
    elif z < 24:
        cat = 3  # Mid-rise upper
    else:
        cat = 4  # High-rise
    
    for rx in range(divisions_x):
        for ry in range(divisions_y):
            x = rx * room_width
            y = ry * room_length
            
            cell = tf.Cell.Box(x, y, z, room_width, room_length, floor_height)
            cells.append(cell)
            cell_categories.append(cat)

# Create the CellComplex
cc = tf.CellComplex.ByCells(cells)

print(f"Building CellComplex created:")
print(f"  Cells: {cc.NumCells()}")
print(f"  Total Volume: {cc.Volume():.1f} m^3")
print(f"  Total Area: {cc.Area():.1f} m^2")

In [ ]:
def show_building_categorized(cells, categories, title="Building"):
    """Visualize building with color-coded cells by category"""
    fig = go.Figure()
    
    # Thermal colorscale colors
    colors = {
        0: '#0d0887',  # Deep blue
        1: '#7201a8',  # Purple
        2: '#bd3786',  # Pink
        3: '#ed7953',  # Orange
        4: '#fdca26'   # Yellow
    }
    
    for i, cell in enumerate(cells):
        cat = categories[i]
        color = colors.get(cat, '#808080')
        
        faces = cell.Faces()
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.5,
                    alphahull=0,
                    showlegend=False
                ))
                
                # Add edges
                for k in range(len(coords)):
                    p1 = coords[k]
                    p2 = coords[(k + 1) % len(coords)]
                    fig.add_trace(go.Scatter3d(
                        x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                        mode='lines',
                        line=dict(color='lightgrey', width=1),
                        showlegend=False
                    ))
    
    # Add legend
    for cat, color in colors.items():
        labels = ['Ground', 'Low-rise', 'Mid-lower', 'Mid-upper', 'High-rise']
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode='markers',
            marker=dict(size=10, color=color),
            name=f'Category {cat}: {labels[cat]}'
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_building_categorized(cells, cell_categories, "Building - Categorized by Height").show()

## 2. Create a Graph from the Building

The dual graph represents rooms as vertices and connections (shared walls/floors) as edges.

In [ ]:
# Create a graph from the CellComplex
g = tf.Graph.ByTopology(cc)

# Get graph vertices and their properties
g_vertices = g.Vertices()

print(f"Graph created:")
print(f"  Vertices: {g.Order()}")
print(f"  Edges: {g.Size()}")
print(f"  Density: {g.Density():.3f}")

In [ ]:
# NOTE: Dictionary operations are not available in topologic_fast
# In topologicpy, we would transfer dictionaries from selectors to cells
# Here we manually assign categories based on vertex Z coordinate

def get_category_from_z(z):
    """Determine category based on Z coordinate"""
    if z < 6:
        return 0
    elif z < 12:
        return 1
    elif z < 18:
        return 2
    elif z < 24:
        return 3
    else:
        return 4

# Assign categories to graph vertices based on their Z coordinate
vertex_categories = []
for v in g_vertices:
    coords = v.Coordinates()
    cat = get_category_from_z(coords[2])
    vertex_categories.append(cat)

# Print category distribution
for cat in range(5):
    count = vertex_categories.count(cat)
    print(f"Category {cat}: {count} vertices")

In [ ]:
def show_graph_categorized(graph, categories, title="Graph"):
    """Visualize graph with color-coded vertices by category"""
    fig = go.Figure()
    
    # Thermal colorscale colors
    colors = {
        0: '#0d0887',
        1: '#7201a8',
        2: '#bd3786',
        3: '#ed7953',
        4: '#fdca26'
    }
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='lightgrey', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices grouped by category
    g_vertices = graph.Vertices()
    
    for cat in range(5):
        x, y, z, labels = [], [], [], []
        for i, v in enumerate(g_vertices):
            if categories[i] == cat:
                coords = v.Coordinates()
                x.append(coords[0])
                y.append(coords[1])
                z.append(coords[2])
                labels.append(f"Cat {cat}")
        
        if x:
            labels_text = ['Ground', 'Low-rise', 'Mid-lower', 'Mid-upper', 'High-rise']
            fig.add_trace(go.Scatter3d(
                x=x, y=y, z=z,
                mode='markers',
                marker=dict(size=6, color=colors[cat]),
                name=f'{labels_text[cat]} (Cat {cat})',
                hovertext=[f'Category: {cat}' for _ in x],
                hoverinfo='text'
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_graph_categorized(g, vertex_categories, "Building Graph - Categorized by Height").show()

## 3. Compute Depth Map

A depth map shows how many steps (edges) it takes to reach each vertex from all other vertices. This is a measure of centrality in the graph.

In [ ]:
# NOTE: Graph.DepthMap is not directly available in topologic_fast
# We compute it manually using shortest path distances

def compute_depthmap(graph):
    """Compute total distance from each vertex to all other vertices"""
    vertices = graph.Vertices()
    n = len(vertices)
    depthmap = []
    
    for i, v in enumerate(vertices):
        total_depth = 0
        for j, other in enumerate(vertices):
            if i != j:
                dist = graph.Distance(v, other)
                if dist is not None:
                    total_depth += dist
        depthmap.append(total_depth)
    
    return depthmap

depthmap = compute_depthmap(g)

print("Depth map computed:")
print(f"  Min depth: {min(depthmap)}")
print(f"  Max depth: {max(depthmap)}")
print(f"  Mean depth: {sum(depthmap)/len(depthmap):.1f}")
print(f"\nDepth values: {depthmap}")

In [ ]:
def show_graph_depthmap(graph, depthmap, title="Graph - Depth Map"):
    """Visualize graph with vertices colored by depth map value"""
    fig = go.Figure()
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='lightgrey', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices colored by depth
    g_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in g_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers+text',
        marker=dict(
            size=10,
            color=depthmap,
            colorscale='Viridis',
            colorbar=dict(title='Total Depth'),
            showscale=True
        ),
        text=[str(d) for d in depthmap],
        textposition='top center',
        textfont=dict(size=8),
        name='Vertices',
        hovertext=[f'Depth: {d}' for d in depthmap],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_graph_depthmap(g, depthmap).show()

## 4. Categorize Faces by Height

We can also visualize the building with faces colored by their height category.

In [ ]:
# Get faces from the CellComplex and categorize them
faces = cc.Faces()

face_categories = []
for face in faces:
    com = face.CenterOfMass()
    cat = get_category_from_z(com[2])
    face_categories.append(cat)

print(f"Total faces: {len(faces)}")
for cat in range(5):
    count = face_categories.count(cat)
    print(f"  Category {cat}: {count} faces")

In [ ]:
def show_faces_categorized(faces, categories, opacity=0.5, title="Faces"):
    """Visualize faces with color based on category"""
    fig = go.Figure()
    
    # Thermal colorscale colors
    colors = {
        0: '#0d0887',
        1: '#7201a8',
        2: '#bd3786',
        3: '#ed7953',
        4: '#fdca26'
    }
    
    for i, face in enumerate(faces):
        cat = categories[i]
        color = colors.get(cat, '#808080')
        
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        if len(coords) >= 3:
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            z = [c[2] for c in coords]
            
            fig.add_trace(go.Mesh3d(
                x=x, y=y, z=z,
                color=color,
                opacity=opacity,
                alphahull=0,
                showlegend=False
            ))
            
            # Add edges
            for k in range(len(coords)):
                p1 = coords[k]
                p2 = coords[(k + 1) % len(coords)]
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='lightgrey', width=1),
                    showlegend=False
                ))
    
    # Add legend
    labels_text = ['Ground', 'Low-rise', 'Mid-lower', 'Mid-upper', 'High-rise']
    for cat, color in colors.items():
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode='markers',
            marker=dict(size=10, color=color),
            name=f'{labels_text[cat]} (Cat {cat})'
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_faces_categorized(faces, face_categories, opacity=0.5, title="Building Faces - Categorized by Height").show()

## 5. Create Graph from Vertices and Edges

We can also create a graph directly from the CellComplex's vertices and edges.

In [ ]:
# NOTE: In topologicpy, this would use CellComplex.Edges and CellComplex.Vertices
# topologic_fast CellComplex doesn't have Edges() method directly
# We can get edges from the faces

# Get all edges from all faces (removing duplicates would be needed for a proper edge list)
all_edges = []
for face in faces:
    face_edges = face.Edges()
    all_edges.extend(face_edges)

print(f"Total edges collected: {len(all_edges)}")

# For demonstration, we'll use a simpler approach - create a grid graph
# Create vertices at grid positions
vertices_grid = []
for floor in range(num_floors + 1):
    for rx in range(divisions_x + 1):
        for ry in range(divisions_y + 1):
            v = tf.Vertex.ByCoordinates(
                rx * room_width,
                ry * room_length,
                floor * floor_height
            )
            vertices_grid.append(v)

print(f"Grid vertices: {len(vertices_grid)}")

In [ ]:
def show_skeleton(cellcomplex, title="Building Skeleton"):
    """Show the edge skeleton of a cell complex"""
    fig = go.Figure()
    
    faces = cellcomplex.Faces()
    drawn_edges = set()
    
    for face in faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        for k in range(len(coords)):
            p1 = coords[k]
            p2 = coords[(k + 1) % len(coords)]
            
            # Create a unique edge key (sorted by coordinates)
            edge_key = tuple(sorted([tuple(p1), tuple(p2)]))
            
            if edge_key not in drawn_edges:
                drawn_edges.add(edge_key)
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='blue', width=2),
                    showlegend=False
                ))
    
    # Add vertices
    vertex_set = set()
    x_list, y_list, z_list = [], [], []
    
    for face in faces:
        vertices = face.Vertices()
        for v in vertices:
            coords = tuple(v.Coordinates())
            if coords not in vertex_set:
                vertex_set.add(coords)
                x_list.append(coords[0])
                y_list.append(coords[1])
                z_list.append(coords[2])
    
    fig.add_trace(go.Scatter3d(
        x=x_list, y=y_list, z=z_list,
        mode='markers',
        marker=dict(size=3, color='red'),
        name='Vertices'
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_skeleton(cc).show()

## Summary

In this tutorial, we learned:

1. **Building Creation** - Created a multi-story building using `tf.Cell.Box()` and `tf.CellComplex.ByCells()`
2. **Categorization** - Assigned categories to cells based on their height
3. **Graph Operations** - Created dual graphs using `tf.Graph.ByTopology()`
4. **Depth Maps** - Computed depth maps to analyze graph centrality
5. **Visualization** - Used Plotly for color-coded 3D visualization

### API Differences from topologicpy

| topologicpy | topologic_fast | Notes |
|-------------|----------------|-------|
| `CellComplex.Prism(width, length, height, wSides=10)` | `tf.CellComplex.ByCells(cells)` | Create cells manually |
| `Dictionary.ByKeyValue(key, value)` | Not available | Manual categorization |
| `Topology.SetDictionary(topo, dict)` | Not available | Store data separately |
| `Topology.TransferDictionariesBySelectors(...)` | Not available | Manual assignment |
| `Graph.DepthMap(graph, vertices)` | Manual computation | Use `graph.Distance()` |
| `Topology.Show(...)` | Use Plotly directly | |
| `Graph.Show(...)` | Use Plotly directly | |

### Features Not Available in topologic_fast

- `Dictionary` class and dictionary operations
- `Topology.TransferDictionariesBySelectors()` - Data must be managed externally
- `Graph.DepthMap()` - Can be computed manually using `Graph.Distance()`
- `CellComplex.Prism()` - Use `Cell.Box()` and `CellComplex.ByCells()` instead